# **Write the submission file `submission.py`**

In [ ]:
%%writefile submission.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import json
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression

def add_columns(x):
  x["x^2"] = x["x"] ** 2
  x["positive"] = x["x"] > 0

  return x

def apply_scaler(x, column_names, training=True):
  scaler = StandardScaler()
  if training:
    scaler.fit(x[column_names])
    joblib.dump(scaler, "scaler.joblib")
  else:
    scaler = joblib.load("scaler.joblib")

  # The output is np array, convert it back to pandas
  scaled_columns_np = scaler.transform(x[column_names])

  output_column_names = scaler.get_feature_names_out()
  x = x.drop(columns = column_names)

  x[output_column_names] = pd.DataFrame(
      scaled_columns_np,
      columns=output_column_names,
      index=x.index
  )

  return x

def apply_onehot(x, column_names, training=True):
  onehot = OneHotEncoder()
  if training:
    onehot.fit(x[column_names])
    joblib.dump(onehot, "onehot.joblib")
  else:
    onehot = joblib.load("onehot.joblib")

  # The output is csr array, convert it back to pandas
  onehot_columns_np = onehot.transform(x[column_names])

  x = x.drop(columns = column_names)
  output_column_names = onehot.get_feature_names_out()
  x[output_column_names] = pd.DataFrame(
      onehot_columns_np.toarray(),
      columns=output_column_names,
      index = x.index
  )

  return x


def feature_eng(x, training=True):
  x = add_columns(x)
  x = apply_scaler(x, ["x", "x^2"], training)
  x = apply_onehot(x, ["positive"], training)
  return x


############################################################
# ⚠️ The grader will call: predict(test_path)
############################################################
def predict(test_path):
    """
    Although we provide a list of permitted packages,
    not all of them are necessarily required,
    nor is the list guaranteed to be sufficient.
    """



    # Load test dataset
    with open(test_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    #############################
    # ENTER YOUR CODE BELOW
    #############################

    # Convert the data into pandas
    df = pd.DataFrame(data)
    print(df.head())

    # Take out X from df
    x_raw = df[["x"]]

    # Perform feature engineering on the input data
    x = feature_eng(x_raw, training=False).to_numpy()

    # Load model from checkpoint (Cheatsheet)
    model = joblib.load("model.joblib")

    return model.predict(x)

Overwriting submission.py


# **Prepare and download the zip file**

In [ ]:
# Upload all files downloaded to your local directory in the training phase

from google.colab import files
uploaded = files.upload()

In [ ]:
# Prepare the zip file

import zipfile

zip_name = "TestID_LastName_FirstName.zip"

with zipfile.ZipFile(zip_name, 'w') as f:
    # Zip files
    f.write("submission.py")
    f.write("model.joblib")
    f.write("onehot.joblib")
    f.write("scaler.joblib")

In [ ]:
# Download the zip file to your local directory

files.download(zip_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>